# 11.h - QDM of tasmax and tasmin: days where the corrected tasmin exceeds the corrected tasmax

**Ad hoc test** on the quantile delta mapping of 11.g, which corrects tasmax and tasmin separately (additive QDM of
each variable, in the window of each indicator, against ERA5 1981-2014). Two separate corrections can invert the
order of the two temperatures on days where they are close (small diurnal range, cloudy humid days). The indicators of
11.g that use one variable only are not affected; the degree-days above 30 degC use both (tasmin is bounded by tasmax
there). This test counts the inversions; it reads the inputs of 11.a / 11.g and writes only in
`test_risk_indicators/<route>/heat_qdm_inversions/`.

**Method.** The QDM of 11.g (same histograms, same transfer, functions verbatim) applied to tasmax and to tasmin in each
window (`WINDOWS`: whole season, 30 days around anthesis, phase 2) of today's cropland of each crop; per cell, the share
of the days of the window with corrected tasmin > corrected tasmax, and the mean size of the inversion; the same for the
raw values (expected 0). Area-weighted (harvested area) globally and in the 6 AR6 regions of 11.a with the largest
area of each crop, per scenario.

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import time
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu
CFG = cu.load_config()

# ---- parameters (as 11.a)
CHOSEN_VARIANT = "A_60d"          # calendar of the ERA5 BIO files (anthesis = start of phase 2)
CELL_AREA_THR = 0.0005
CROPS = ["Maize_rf", "Potato_rf", "Soybean_rf", "Rice_rf", "Wheat_rf", "Wheat-winter_rf", "Wheat-spring_rf"]
ERA5_NAME, ERA5_PERIOD = "ERA5", ("1981", "2014")    # BIO files of 1.1.d (calendar)
REF, REF_PERIOD = "gcm_historical", ("1981", "2014")  # EC-Earth3 historical, raw
RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]
RUN_PERIOD = {}                   # run -> (first, last year) to read only part of a file; whole file by default
PAIRS = [("gcm_2025_hosing", "gcm_2025_nohosing"), ("gcm_2050_hosing", "gcm_2050_nohosing")]   # hosing - original
HEAT_TCRIT = {"Maize": 35.0, "Wheat": 34.0, "Rice": 36.0, "Soybean": 39.0, "Potato": 30.0}   # degC, 11.a / 11.b
PHYSIO = {                        # 11.c: (name, variable, window, kind, threshold degC)
    "Maize": [("sterility (Tmax > 36 degC, anthesis)", "tasmax", "flower", "days", 36.0),
              ("yield heat (degree-days > 30 degC, season)", "tasmax", "season", "kdd", 30.0)],
    "Soybean": [("yield heat (degree-days > 30 degC, season)", "tasmax", "season", "kdd", 30.0)],
    "Rice": [("sterility (Tmax > 38 degC, anthesis)", "tasmax", "flower", "days", 38.0),
             ("warm nights (mean Tmin, grain filling)", "tasmin", "p2", "mean", None)],
    "Wheat": [("sterility (Tmax > 31 degC, anthesis)", "tasmax", "flower", "days", 31.0),
              ("senescence (Tmax > 34 degC, grain filling)", "tasmax", "p2", "days", 34.0)],
    "Potato": [("warm nights (Tmin > 20 degC, tuber bulking)", "tasmin", "tuber", "days", 20.0)],
}
FLOWER_WINDOW = (-15, 15)         # days around anthesis (start of phase 2): 30 days, Gourdji et al. 2013
TUBER_START = 50                  # potato: tuber initiation, days after sowing (variant C_crop of 1.1.d)
HEAT_PCT = 75                     # event: indicator above this percentile of the reference seasons
N_REGIONS = 6                     # AR6 land regions with the largest area of the crop
TAIL_Q = 0.01                     # QM: beyond the 1st / 99th percentile of ERA5, shift by the difference of that percentile (11.b)
BIN = (-60.0, 70.0, 0.1)          # degC: bins of the histograms (11.a)
METHODS = ("QM", "QDM")

ROUTE = cu.route(CELL_AREA_THR)
TAG = f"{ERA5_NAME}_{ERA5_PERIOD[0]}_{ERA5_PERIOD[1]}_ecearth3"
ERA5_DIR = cu.out_dir(CFG, "1_biovariables", f"{ERA5_NAME}_{ROUTE}", CHOSEN_VARIANT)
OUT = cu.out_dir(CFG, "test_risk_indicators", ROUTE, "heat_qdm_inversions")   # the only folder written
WINDOWS = ("season", "flower", "p2")  # windows of 11.g where tasmax and tasmin are both corrected
SCEN = ["historical", *RUNS]
SPEC = {"historical": (REF, REF_PERIOD), **{r: (r, RUN_PERIOD.get(r)) for r in RUNS}}

GRID = cu.svm_grid(CFG)
LAND = cu.land_mask(CFG).values
base_of = lambda crop: crop.split("_")[0].split("-")[0]
# indicators of each crop: the heat of 11.b, then the physiological ones of 11.c
HEAT = {c: [("heat (Tmax > Tcrit, phase 2)", "tasmax", "p2", "days", HEAT_TCRIT[base_of(c)])] + PHYSIO[base_of(c)]
        for c in CROPS}
# (variable, window) pairs whose daily values are corrected, per crop (degree-days: tasmax and tasmin)
PAIRS_VW = {c: [(v, w) for w in WINDOWS for v in ("tasmax", "tasmin")] for c in CROPS}

# today's cropland of each crop (as 11.a, RISK_DOMAIN = "cropland"): harvested area, calendar
CROP = {}
for crop in CROPS:
    cm = cu.crop_mask(CFG, crop)
    with xr.open_dataset(ERA5_DIR / f"BIOvariables_{TAG}_{crop}.nc") as ds:
        cal = {v: ds[v].values for v in ("sowing_doy", "phase2_start_doy", "maturity_doy")}
    known = LAND & np.isfinite(cal["sowing_doy"]) & np.isfinite(cal["maturity_doy"])
    pres = known & (cm["presence"].values == 1)
    if CELL_AREA_THR:
        pres &= cu.cell_area_frac(CFG, cm["area_ha"].values) > CELL_AREA_THR
    CROP[crop] = {"mask": pres, "area": np.nan_to_num(cm["area_ha"].values), "cal": cal}
UNION = np.logical_or.reduce([CROP[c]["mask"] for c in CROPS])
UI, UJ = np.nonzero(UNION)
COL = -np.ones(LAND.shape, int); COL[UI, UJ] = np.arange(UI.size)   # column of each cell in the daily arrays
for crop in CROPS:
    i, j = np.nonzero(CROP[crop]["mask"])
    CROP[crop].update({"ij": (i, j), "col": COL[i, j], "w": CROP[crop]["area"][i, j],
                       **{k: CROP[crop]["cal"][k][i, j].astype(float) for k in CROP[crop]["cal"]}})

try:                                         # AR6 land regions (Iturbide et al. 2020) of the cells
    import regionmask
    AR6 = regionmask.defined_regions.ar6.land
    REG = AR6.mask(GRID["lon"], GRID["lat"]).values
    REG_NAME = dict(zip(AR6.numbers, AR6.abbrevs))
except Exception as e:                       # no regions: global results only
    print("regionmask not available:", e)
    REG, REG_NAME = np.full(LAND.shape, np.nan), {}
print(f"{len(CROPS)} crops, {UI.size} cells | reference {REF} {REF_PERIOD} | runs {RUNS} | written only in {OUT}")
for c in CROPS:
    print(f"  {c:16s} " + " | ".join(h[0] for h in HEAT[c]) + f" | corrected: {PAIRS_VW[c]}")

## 1. Windows, histograms, threshold mapping (functions of 11.a) and QDM from the histograms

In [ ]:
def degree_days_above(tx, tn, b):
    # degree-days above b of one day, single-sine curve between Tmin and Tmax (Baskerville & Emin 1969) (11.a)
    m, a = (tx + tn) / 2, np.maximum((tx - tn) / 2, 1e-6)
    th = np.arcsin(np.clip((b - m) / a, -1, 1))
    return np.maximum(((m - b) * (np.pi / 2 - th) + a * np.cos(th)) / np.pi, 0.0)


def ylen(y):
    return 366 if (y % 4 == 0 and (y % 100 != 0 or y % 400 == 0)) else 365


def windows(dates, c):
    # the seasons of the crop (harvest year hy, with hy - 1 in the data): day indices and masks (days, cells) of the
    # season, phase 2, the 30 days around anthesis, tuber bulking (11.a)
    yr, doy = dates.year.values, dates.dayofyear.values
    years = sorted(set(yr))
    sow, ant, mat = c["sowing_doy"], c["phase2_start_doy"], c["maturity_doy"]
    wrap = mat > 365
    for hy in years:
        if hy - 1 not in years:
            continue
        idx = np.where((yr == hy) | (yr == hy - 1))[0]
        T = (doy[idx] + np.where(yr[idx] == hy, ylen(hy - 1), 0))[:, None]
        shift = np.where(wrap, 0, ylen(hy - 1))[None]
        s, a, m = sow[None] + shift, ant[None] + shift, mat[None] + shift
        yield hy, idx, {"season": (T >= s) & (T <= m), "p2": (T >= a) & (T <= m),
                        "flower": (T >= a + FLOWER_WINDOW[0]) & (T < a + FLOWER_WINDOW[1]),
                        "tuber": (T >= s + TUBER_START) & (T <= m)}


NB = int(round((BIN[1] - BIN[0]) / BIN[2]))
CENTRE = BIN[0] + (np.arange(NB) + 0.5) * BIN[2]


def bin_of(x):
    return np.clip(((x - BIN[0]) / BIN[2]).astype(int), 0, NB - 1)


def counts(X, dates, c, wnd):
    # histogram (cells, bins) of the daily values of X in the window, over all the seasons (11.a)
    n, col = c["col"].size, c["col"]
    cnt = np.zeros(n * NB, np.int64)
    for _, idx, W in windows(dates, c):
        b = bin_of(X[idx][:, col])
        cnt += np.bincount((np.arange(n)[None] * NB + b)[W[wnd]], minlength=n * NB)
    return cnt.reshape(n, NB)


def cdf(cnt):
    return np.cumsum(cnt, 1) / np.maximum(cnt.sum(1, keepdims=True), 1)


def value_at(cnt, q):
    # value at the quantile q (per cell): upper edge of the first bin where the empirical CDF reaches q (11.a)
    return BIN[0] + (np.argmax(cdf(cnt) >= np.asarray(q)[:, None] - 1e-12, axis=1) + 1) * BIN[2]


def mapped_threshold(cnt_e, cnt_m, t0):
    # 11.b: threshold of EC-Earth3 = its quantile q of the window, q = quantile of t0 in ERA5; beyond the 1st / 99th
    # percentile of ERA5: t0 shifted by the difference of that percentile (EC-Earth3 - ERA5)
    n = cnt_e.shape[0]
    q = cdf(cnt_e)[np.arange(n), int(np.clip((t0 - BIN[0]) / BIN[2], 0, NB - 1))]
    thr = value_at(cnt_m, np.clip(q, TAIL_Q, 1 - TAIL_Q))
    for tail, sel in ((1 - TAIL_Q, q > 1 - TAIL_Q), (TAIL_Q, q < TAIL_Q)):
        qq = np.full(n, tail)
        thr = np.where(sel, t0 + value_at(cnt_m, qq) - value_at(cnt_e, qq), thr)
    return thr, q


def quantile(C, tau):
    # quantile function of one cell from its CDF at the upper bin edges (C, NB), linear within the bin
    b = np.minimum(np.searchsorted(C, tau, side="left"), NB - 1)
    lo = np.where(b > 0, C[np.maximum(b - 1, 0)], 0.0)
    frac = np.where(C[b] > lo, (tau - lo) / np.maximum(C[b] - lo, 1e-12), 1.0)
    return BIN[0] + (b + np.clip(frac, 0, 1)) * BIN[2]


def qdm_table(cnt_e, cnt_h, cnt_r):
    # QDM (Cannon et al. 2015, additive) of the run r against ERA5, calibrated on the historical run h, per cell: the
    # correction Q_e(tau) - Q_h(tau) at the centre of each bin of the run, tau = mid-rank of the bin in the run
    # (cells, NB); bins absent from the run take the correction of the nearest bin of the run (only used between two
    # bin centres, see apply_table)
    Ce, Ch, Cr = cdf(cnt_e), cdf(cnt_h), cdf(cnt_r)
    corr = np.zeros(cnt_r.shape, np.float32)
    for k in range(cnt_r.shape[0]):
        used = np.nonzero(cnt_r[k])[0]
        if used.size == 0 or cnt_e[k].sum() == 0 or cnt_h[k].sum() == 0:
            continue
        tau = (np.where(used > 0, Cr[k, np.maximum(used - 1, 0)], 0.0) + Cr[k, used]) / 2
        corr[k] = np.interp(np.arange(NB), used, quantile(Ce[k], tau) - quantile(Ch[k], tau))
    return corr


def apply_table(corr, x):
    # corrected daily values (days, cells): x + the correction of the cell interpolated linearly between bin centres
    pos = (x - BIN[0]) / BIN[2] - 0.5
    b0 = np.clip(np.floor(pos).astype(int), 0, NB - 2)
    f = np.clip(pos - b0, 0.0, 1.0)
    c = np.arange(x.shape[1])[None]
    return x + corr[c, b0] * (1 - f) + corr[c, b0 + 1] * f

## 2. Inversions per cell, window and scenario

In [ ]:
def read_daily(name, period, vars_=("tasmax", "tasmin")):
    t0 = time.time()
    arr = {}
    for v in vars_:
        da = cu.read_climate(CFG, name, v, GRID, period)
        arr[v] = cu.land_series(da, (UI, UJ))
    dates = pd.DatetimeIndex(da["time"].values).normalize()
    print(f"  {name} {period or 'whole file'}: {dates[0].date()} -> {dates[-1].date()} ({(time.time() - t0) / 60:.1f} min)", flush=True)
    return arr, dates


def histograms(arr, dates):
    return {(c, v, w): counts(arr[v], dates, CROP[c], w) for c in CROPS for v, w in PAIRS_VW[c]}


def inversions(arr, dates, c, luts, wnd):
    # per cell: days of the window, days with corrected tasmin > corrected tasmax, sum of the excess, raw inversions
    n = c["col"].size
    days, inv, exc, raw = (np.zeros(n) for _ in range(4))
    for _, idx, W in windows(dates, c):
        m = W[wnd]
        tx, tn = arr["tasmax"][idx][:, c["col"]], arr["tasmin"][idx][:, c["col"]]
        txc, tnc = apply_table(luts[("tasmax", wnd)], tx), apply_table(luts[("tasmin", wnd)], tn)
        d = np.where(m, tnc - txc, 0.0)
        days += m.sum(0); inv += (d > 0).sum(0); exc += np.where(d > 0, d, 0.0).sum(0); raw += ((tn > tx) & m).sum(0)
    return days, inv, exc, raw


arr, dates = read_daily("era5", ERA5_PERIOD)
CNT_E = histograms(arr, dates)
del arr
CNT_H, RES = None, {}
for s in SCEN:
    name, period = SPEC[s]
    arr, dates = read_daily(name, period)
    cnt = histograms(arr, dates)
    if s == "historical":
        CNT_H = cnt
    for crop in CROPS:
        luts = {(v, w): qdm_table(CNT_E[(crop, v, w)], CNT_H[(crop, v, w)], cnt[(crop, v, w)]) for v, w in PAIRS_VW[crop]}
        for wnd in WINDOWS:
            RES[(s, crop, wnd)] = inversions(arr, dates, CROP[crop], luts, wnd)
    del arr
    print(f"{s}: done", flush=True)

## 3. Share of the days with tasmin > tasmax after the QDM, per zone

In [ ]:
def zones_of(crop):
    # (name, cells, weights): today's cropland globally and in its N_REGIONS largest AR6 regions (harvested area) (11.a)
    c = CROP[crop]
    i, j = c["ij"]
    reg = REG[i, j]
    tot = pd.Series(c["w"]).groupby(reg).sum().sort_values(ascending=False) if REG_NAME else pd.Series(dtype=float)
    top = [int(r) for r in tot.index[:N_REGIONS] if np.isfinite(r)]
    return [("global", np.ones(i.size, bool), c["w"])] + [(REG_NAME[r], reg == r, c["w"]) for r in top]


rows = []
for (s, crop, wnd), (days, inv, exc, raw) in RES.items():
    share = np.where(days > 0, inv / np.maximum(days, 1), np.nan)
    for zname, z, w in zones_of(crop):
        ok = z & (days > 0)
        if not ok.any():
            continue
        rows.append({"crop": crop, "window": wnd, "scenario": s, "zone": zname,
                     "inverted days (%)": 100 * inv[ok].sum() / days[ok].sum(),
                     "mean inversion (degC)": exc[ok].sum() / max(inv[ok].sum(), 1),
                     "raw inverted days (%)": 100 * raw[ok].sum() / days[ok].sum(),
                     "area with > 1 % inverted (%)": 100 * w[ok & (share > 0.01)].sum() / w[ok].sum(),
                     "area with > 5 % inverted (%)": 100 * w[ok & (share > 0.05)].sum() / w[ok].sum()})
INV = pd.DataFrame(rows).set_index(["crop", "window", "zone", "scenario"]).sort_index()
INV.to_csv(OUT / "tasmin_above_tasmax_after_qdm.csv")
with pd.option_context("display.max_rows", 400, "display.width", 250, "display.float_format", "{:.3f}".format):
    display(INV.xs("global", level="zone"))
    display(INV.sort_values("inverted days (%)", ascending=False).head(30))

## 4. Map (maize, whole season, 2050 hosing)

In [ ]:
crop = "Maize_rf" if "Maize_rf" in CROPS else CROPS[0]
i, j = CROP[crop]["ij"]
fig, axes = plt.subplots(1, 2, figsize=(22, 6))
for ax, s in zip(axes, ("historical", PAIRS[-1][0])):
    days, inv, exc, raw = RES[(s, crop, "season")]
    g = np.full(LAND.shape, np.nan); g[i, j] = 100 * inv / np.maximum(days, 1)
    ax.pcolormesh(GRID["lon"], GRID["lat"], np.where(LAND, 0, np.nan), cmap="Greys", vmin=0, vmax=4, shading="nearest")
    im = ax.pcolormesh(GRID["lon"], GRID["lat"], g, cmap="magma_r", vmin=0, vmax=max(1, np.nanpercentile(g, 99)), shading="nearest")
    ax.set_ylim(-60, 75); ax.set_title(f"{crop}, {s}: season days with corrected tasmin > tasmax (%)", fontsize=10)
    plt.colorbar(im, ax=ax, shrink=0.8)
fig.savefig(OUT / f"tasmin_above_tasmax_{crop}.png", dpi=90, bbox_inches="tight"); plt.show()